# Patient Journey & Treatment Adoption Analytics
## Understanding Patient Drop-offs, Treatment Adoption, Patient Segments and Market Opportunities

---

**Company:** NovaCure Pharmaceuticals *(fictional)*  
**Treatment:** Therapy X *(fictional)*  
**Dataset:** 25,000 synthetic patients  
**Analysis Period:** 2021–2023  

---

> **DISCLAIMER:** This notebook uses synthetic data created for analytical and educational purposes only.
> It does not represent real patients, real clinical outcomes, medical advice, or actual pharmaceutical market data.

---

## Notebook Sections

1. Business Problem  
2. Import Libraries  
3. Load Data  
4. Data Overview  
5. Exploratory Data Analysis  
6. Patient Journey Analysis  
7. Patient Segmentation  
8. Treatment Adoption Analysis  
9. Adoption Drivers (Logistic Regression)  
10. Market Opportunity Analysis  
11. Conclusions


---
## 1. Business Problem

NovaCure Pharmaceuticals has launched **Therapy X**, a new treatment for a chronic condition
affecting millions of Americans. Despite wide availability, internal data shows that a
significant portion of diagnosed patients never start the treatment.

The commercial and medical affairs teams need to understand:

1. **Where do patients drop off** during their treatment journey?
2. **Which patient groups** have higher treatment adoption?
3. **What factors** are associated with adoption?
4. **What are the main barriers** to starting treatment?
5. **Which regions** have the highest opportunity for growth?
6. **What should the company prioritise** to improve adoption?

This analysis provides data-driven answers to these questions using a synthetic patient dataset.


---
## 2. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, classification_report)

# ── Plot settings ──────────────────────────────────────────────────────────
PALETTE = "#264653 #2a9d8f #e9c46a #f4a261 #e76f51".split()
BLUE, ORANGE, RED, DARK = "#2a9d8f", "#f4a261", "#e76f51", "#264653"

sns.set_theme(style="whitegrid", font_scale=1.05)
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.bbox": "tight",
    "axes.spines.top": False,
    "axes.spines.right": False,
})

print("✓ All libraries imported successfully")


---
## 3. Load Data

In [ ]:
# If you haven't run the pipeline yet, run:
# python src/run_project.py

df = pd.read_csv("data/patient_data_clean.csv")
print(f"Dataset loaded: {len(df):,} rows × {df.shape[1]} columns")


---
## 4. Data Overview

In [ ]:
# First 5 rows
df.head()


In [ ]:
# Column types and non-null counts
df.info()


In [ ]:
# Statistical summary for numeric columns
df.describe().round(2)


In [ ]:
# Check for missing values
missing = df.isnull().sum()
print("Missing values per column:")
print(missing[missing > 0] if missing.any() else "No missing values found ✓")


In [ ]:
# Distribution of key categorical columns
for col in ["Region", "Insurance_Status", "Disease_Severity", "Drop_Off_Stage"]:
    print(f"\n── {col} ──")
    print(df[col].value_counts())


---
## 5. Exploratory Data Analysis

Before answering specific business questions, we explore the dataset to understand
distributions and relationships.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# Age distribution
axes[0,0].hist(df['Age'], bins=20, color=BLUE, edgecolor='white')
axes[0,0].set_title('Age Distribution')
axes[0,0].set_xlabel('Age')
axes[0,0].set_ylabel('Count')

# Insurance status
ins_counts = df['Insurance_Status'].value_counts()
axes[0,1].bar(ins_counts.index, ins_counts.values, color=PALETTE[:3], edgecolor='white')
axes[0,1].set_title('Insurance Status Distribution')
axes[0,1].set_ylabel('Count')

# Disease severity
sev_counts = df['Disease_Severity'].value_counts()
axes[0,2].bar(sev_counts.index, sev_counts.values, color=[BLUE, ORANGE, RED], edgecolor='white')
axes[0,2].set_title('Disease Severity Distribution')
axes[0,2].set_ylabel('Count')

# Affordability score
axes[1,0].hist(df['Affordability_Score'], bins=20, color=ORANGE, edgecolor='white')
axes[1,0].set_title('Affordability Score Distribution')
axes[1,0].set_xlabel('Score (0-10)')

# Treatment cost
axes[1,1].hist(df['Treatment_Cost'], bins=20, color=RED, edgecolor='white')
axes[1,1].set_title('Treatment Cost Distribution')
axes[1,1].set_xlabel('Cost (USD)')

# Region distribution
reg_counts = df['Region'].value_counts()
axes[1,2].bar(reg_counts.index, reg_counts.values, color=sns.color_palette(PALETTE), edgecolor='white')
axes[1,2].set_title('Patient Distribution by Region')
axes[1,2].set_ylabel('Count')
plt.xticks(rotation=15)

fig.suptitle('Exploratory Data Analysis — Key Distributions', fontsize=14, fontweight='bold')
fig.tight_layout()
plt.show()


In [ ]:
# Overall adoption rate
adoption_rate = df['Treatment_Started'].mean()
print(f"Overall Treatment Adoption Rate: {adoption_rate:.1%}")
print(f"Patients who started treatment:  {df['Treatment_Started'].sum():,}")
print(f"Patients who did NOT start:      {(df['Treatment_Started'] == 0).sum():,}")


---
## 6. Patient Journey Analysis

The patient journey describes the steps a patient takes from diagnosis to completing follow-up.
We use a **funnel analysis** to see how many patients make it through each stage.


In [ ]:
# Build the funnel
stages = [
    ("Diagnosed",              len(df)),
    ("Doctor Consultation",    df["Doctor_Consultation"].sum()),
    ("Treatment Recommended",  df["Treatment_Recommended"].sum()),
    ("Treatment Started",      df["Treatment_Started"].sum()),
    ("Treatment Continued",    df["Treatment_Continued"].sum()),
    ("Follow-up Completed",    df["Follow_Up_Completed"].sum()),
]

journey = pd.DataFrame(stages, columns=["Stage", "Patients"])
journey["Conversion_%"]  = (journey["Patients"] / journey.loc[0, "Patients"] * 100).round(1)
journey["Drop_Off_%"]    = (
    (journey["Patients"].shift(1) - journey["Patients"]) /
     journey["Patients"].shift(1) * 100
).round(1).fillna(0)

print(journey.to_string(index=False))


In [ ]:
# Patient Journey Funnel Chart
fig, ax = plt.subplots(figsize=(10, 6))
colors = [BLUE if i == 0 else RED if i == 3 else DARK for i in range(len(journey))]

bars = ax.barh(journey["Stage"][::-1], journey["Patients"][::-1],
               color=colors[::-1], edgecolor='white', height=0.6)

for bar, (_, row) in zip(bars, journey[::-1].iterrows()):
    ax.text(bar.get_width() + 100, bar.get_y() + bar.get_height() / 2,
            f"{int(row['Patients']):,}  ({row['Conversion_%']:.1f}%)",
            va='center', fontsize=9)

ax.set_xlabel("Number of Patients")
ax.set_title("Patient Journey Funnel — Therapy X", fontsize=14, fontweight='bold')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax.set_xlim(0, journey["Patients"].max() * 1.22)
fig.tight_layout()
plt.show()


In [ ]:
# Where is the biggest drop-off?
biggest = journey.loc[journey["Drop_Off_%"].idxmax()]
print(f"Biggest drop-off stage: {biggest['Stage']}")
print(f"Drop-off rate:          {biggest['Drop_Off_%']:.1f}%")
print(f"Patients lost:          {int(journey['Patients'].shift(1)[journey['Drop_Off_%'].idxmax()] - biggest['Patients']):,}")


---
## 7. Patient Segmentation

We create simple, business-friendly segments based on **clinical need** and **financial ability**.

- **Need Level:** High-Need (Moderate/Severe) vs Low-Need (Mild)
- **Ability Level:** High-Ability (Insured + Affordability ≥ 5), Low-Ability (Uninsured or Affordability < 3), Medium-Ability (all others)


In [ ]:
df2 = df.copy()

# Need Level
df2["Need_Level"] = df2["Disease_Severity"].map(
    {"Severe": "High-Need", "Moderate": "High-Need", "Mild": "Low-Need"}
)

# Ability Level
df2["Ability_Level"] = np.where(
    (df2["Insurance_Status"] == "Insured") & (df2["Affordability_Score"] >= 5),
    "High-Ability",
    np.where(
        (df2["Insurance_Status"] == "Uninsured") | (df2["Affordability_Score"] < 3),
        "Low-Ability",
        "Medium-Ability"
    )
)

df2["Business_Segment"] = df2["Need_Level"] + " / " + df2["Ability_Level"]

seg = (
    df2.groupby("Business_Segment")
    .agg(Total=("Patient_ID", "count"),
         Adopted=("Treatment_Started", "sum"),
         Adoption_Rate=("Treatment_Started", "mean"))
    .reset_index()
    .sort_values("Total", ascending=False)
)
seg["Adoption_%"] = (seg["Adoption_Rate"] * 100).round(1)
seg["Untreated"] = seg["Total"] - seg["Adopted"]
print(seg[["Business_Segment", "Total", "Adopted", "Untreated", "Adoption_%"]].to_string(index=False))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Segment size
palette = sns.color_palette(PALETTE, len(seg))
axes[0].bar(seg["Business_Segment"], seg["Total"], color=palette, edgecolor='white', width=0.55)
axes[0].set_title("Patient Segment Distribution", fontweight='bold')
axes[0].set_ylabel("Number of Patients")
axes[0].tick_params(axis='x', rotation=20)
for bar in axes[0].patches:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                 f"{int(bar.get_height()):,}", ha='center', fontsize=8)

# Adoption by segment
colors2 = [BLUE if "High-Need" in s else ORANGE for s in seg["Business_Segment"]]
axes[1].bar(seg["Business_Segment"], seg["Adoption_%"], color=colors2, edgecolor='white', width=0.55)
axes[1].set_title("Adoption Rate by Segment", fontweight='bold')
axes[1].set_ylabel("Adoption Rate (%)")
axes[1].tick_params(axis='x', rotation=20)
axes[1].set_ylim(0, seg["Adoption_%"].max() * 1.15)
for bar, val in zip(axes[1].patches, seg["Adoption_%"]):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f"{val:.1f}%", ha='center', fontsize=9, fontweight='bold')

fig.suptitle("Patient Segmentation Analysis", fontsize=14, fontweight='bold')
fig.tight_layout()
plt.show()


---
## 8. Treatment Adoption Analysis

We compare adoption rates across multiple dimensions to identify which groups adopt
Therapy X most — and least — readily.


In [ ]:
def adoption_by(col, order=None):
    result = (
        df.groupby(col)["Treatment_Started"]
        .agg(["sum", "count", "mean"])
        .reset_index()
    )
    result.columns = [col, "Adopted", "Total", "Rate"]
    result["Rate_%"] = (result["Rate"] * 100).round(1)
    if order:
        result = result.set_index(col).loc[order].reset_index()
    return result

# Adoption by severity
print("── Disease Severity ──")
print(adoption_by("Disease_Severity", ["Mild","Moderate","Severe"]))

print("\n── Insurance Status ──")
print(adoption_by("Insurance_Status"))

print("\n── Region ──")
print(adoption_by("Region"))

print("\n── Treatment Recommended ──")
rec = adoption_by("Treatment_Recommended")
rec["Treatment_Recommended"] = rec["Treatment_Recommended"].map({1: "Recommended", 0: "Not Recommended"})
print(rec)

print("\n── Cost Band ──")
cost_order = ["Low (<$15k)", "Medium ($15-35k)", "High ($35-60k)", "Very High (>$60k)"]
print(adoption_by("Cost_Band", cost_order))


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

def bar_ax(ax, data, x_col, y_col, title, colors=BLUE):
    if isinstance(colors, str):
        colors = [colors] * len(data)
    bars = ax.bar(data[x_col], data[y_col], color=colors, edgecolor='white', width=0.55)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f"{bar.get_height():.1f}%", ha='center', fontsize=8.5, fontweight='bold')
    ax.set_title(title, fontweight='bold', fontsize=10)
    ax.set_ylabel("Adoption Rate (%)")
    ax.set_ylim(0, data[y_col].max() * 1.2)
    ax.tick_params(axis='x', rotation=15)

sev = adoption_by("Disease_Severity", ["Mild","Moderate","Severe"])
bar_ax(axes[0,0], sev, "Disease_Severity", "Rate_%", "By Disease Severity",
       colors=[BLUE, ORANGE, RED])

ins = adoption_by("Insurance_Status")
bar_ax(axes[0,1], ins, "Insurance_Status", "Rate_%", "By Insurance Status",
       colors=[BLUE, ORANGE, RED])

reg = adoption_by("Region").sort_values("Rate_%", ascending=False)
bar_ax(axes[0,2], reg, "Region", "Rate_%", "By Region",
       colors=sns.color_palette(PALETTE, len(reg)))

cost = adoption_by("Cost_Band", ["Low (<$15k)","Medium ($15-35k)","High ($35-60k)","Very High (>$60k)"])
bar_ax(axes[1,0], cost, "Cost_Band", "Rate_%", "By Treatment Cost Band",
       colors=[BLUE, BLUE, ORANGE, RED])

rec = adoption_by("Treatment_Recommended")
rec["Treatment_Recommended"] = rec["Treatment_Recommended"].map({1: "Recommended", 0: "Not Recommended"})
bar_ax(axes[1,1], rec, "Treatment_Recommended", "Rate_%",
       "By Physician Recommendation", colors=[BLUE, ORANGE])

age = adoption_by("Age_Group")
bar_ax(axes[1,2], age, "Age_Group", "Rate_%", "By Age Group",
       colors=sns.color_palette(PALETTE, len(age)))

fig.suptitle("Treatment Adoption Analysis — Therapy X", fontsize=14, fontweight='bold')
fig.tight_layout()
plt.show()


---
## 9. Adoption Drivers — Logistic Regression

### What is Logistic Regression?
Logistic regression is a simple model that predicts the probability of a **yes/no** outcome —
in this case, whether a patient starts treatment (1) or not (0).

It assigns a **coefficient (weight)** to each input factor:
- **Positive coefficient** → the factor is associated with *higher* likelihood of adoption
- **Negative coefficient** → the factor is associated with *lower* likelihood of adoption

> ⚠️ **Important:** We are identifying *associations*, not causes.
> Correlation does not imply causation.


In [ ]:
df3 = df.copy()
df3["Insurance_Insured"]  = (df3["Insurance_Status"]  == "Insured").astype(int)
df3["Severity_Severe"]    = (df3["Disease_Severity"]  == "Severe").astype(int)
df3["Severity_Moderate"]  = (df3["Disease_Severity"]  == "Moderate").astype(int)

FEATURES = [
    "Age", "Severity_Severe", "Severity_Moderate", "Insurance_Insured",
    "Treatment_Recommended", "Affordability_Score",
    "Healthcare_Access_Score", "Awareness_Score",
    "Previous_Treatment", "Side_Effect_Concern",
]

LABELS = {
    "Age":                     "Age",
    "Severity_Severe":         "Severe Disease",
    "Severity_Moderate":       "Moderate Disease",
    "Insurance_Insured":       "Insured",
    "Treatment_Recommended":   "Physician Recommendation",
    "Affordability_Score":     "Affordability Score",
    "Healthcare_Access_Score": "Healthcare Access",
    "Awareness_Score":         "Awareness Score",
    "Previous_Treatment":      "Prior Treatment",
    "Side_Effect_Concern":     "Side Effect Concern",
}

X = df3[FEATURES].fillna(0)
y = df3["Treatment_Started"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

model = LogisticRegression(max_iter=500, random_state=42)
model.fit(X_train_s, y_train)

y_pred = model.predict(X_test_s)
y_prob = model.predict_proba(X_test_s)[:, 1]

print("── Model Performance ─────────────────────────────")
print(f"  Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"  Precision: {precision_score(y_test, y_pred):.4f}")
print(f"  Recall:    {recall_score(y_test, y_pred):.4f}")
print(f"  F1 Score:  {f1_score(y_test, y_pred):.4f}")
print(f"  ROC-AUC:   {roc_auc_score(y_test, y_prob):.4f}")


In [ ]:
# Feature Importance Chart
coef_df = pd.DataFrame({
    "Feature": FEATURES,
    "Label":   [LABELS[f] for f in FEATURES],
    "Coefficient": model.coef_[0]
}).sort_values("Coefficient")

colors_coef = [RED if c < 0 else BLUE for c in coef_df["Coefficient"]]

fig, ax = plt.subplots(figsize=(9, 5.5))
ax.barh(coef_df["Label"], coef_df["Coefficient"], color=colors_coef, edgecolor='white', height=0.65)
ax.axvline(0, color='#555555', linewidth=0.8, linestyle='--')
ax.set_xlabel("Logistic Regression Coefficient (Standardised)", fontsize=11)
ax.set_title("Factors Associated with Treatment Adoption\n(Logistic Regression Coefficients)",
             fontsize=13, fontweight='bold')
ax.text(0.98, 0.03,
        "Positive = more likely to start treatment\nNegative = less likely",
        transform=ax.transAxes, ha='right', fontsize=8.5, color='#555555')
fig.tight_layout()
plt.show()

print("\nTop POSITIVE factors (most associated with adoption):")
print(coef_df.tail(3)[["Label", "Coefficient"]].to_string(index=False))
print("\nTop NEGATIVE factors (most associated with non-adoption):")
print(coef_df.head(3)[["Label", "Coefficient"]].to_string(index=False))


---
## 10. Market Opportunity Analysis

We identify which regions present the largest commercial opportunity by combining:
1. **Untreated patient volume** (how many patients haven't started?)
2. **Adoption gap** (how far is the region from full adoption?)

Regions with both a large untreated population *and* a large adoption gap score highest.


In [ ]:
mkt = (
    df.groupby("Region")
    .agg(Total=("Patient_ID", "count"),
         Treated=("Treatment_Started", "sum"))
    .reset_index()
)
mkt["Untreated"]      = mkt["Total"] - mkt["Treated"]
mkt["Adoption_Rate_%"] = (mkt["Treated"] / mkt["Total"] * 100).round(2)
mkt["Adoption_Gap_%"]  = (100 - mkt["Adoption_Rate_%"]).round(2)

def norm(x):
    return (x - x.min()) / (x.max() - x.min() + 1e-9)

mkt["Opportunity_Score"] = (0.5 * norm(mkt["Untreated"]) + 0.5 * norm(mkt["Adoption_Gap_%"])).round(4)
mkt = mkt.sort_values("Opportunity_Score", ascending=False)

print(mkt.to_string(index=False))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Untreated patients
palette = sns.color_palette(PALETTE, len(mkt))
bars1 = axes[0].bar(mkt["Region"], mkt["Untreated"], color=palette, edgecolor='white', width=0.55)
for bar in bars1:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
                 f"{int(bar.get_height()):,}", ha='center', fontsize=9, fontweight='bold')
axes[0].set_title("Untreated Patients by Region", fontweight='bold')
axes[0].set_ylabel("Untreated Patients")

# Opportunity score
opp_colors = [RED if i == 0 else ORANGE if i == 1 else BLUE for i in range(len(mkt))]
bars2 = axes[1].bar(mkt["Region"], mkt["Opportunity_Score"], color=opp_colors, edgecolor='white', width=0.55)
for bar in bars2:
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 f"{bar.get_height():.3f}", ha='center', fontsize=9, fontweight='bold')
axes[1].set_title("Market Opportunity Score by Region\n(Higher = More Opportunity)", fontweight='bold')
axes[1].set_ylabel("Opportunity Score (0-1)")

fig.suptitle("Regional Market Opportunity Analysis", fontsize=14, fontweight='bold')
fig.tight_layout()
plt.show()

print(f"\nTop Opportunity Region: {mkt.iloc[0]['Region']}")
print(f"  Untreated patients:    {int(mkt.iloc[0]['Untreated']):,}")
print(f"  Adoption rate:         {mkt.iloc[0]['Adoption_Rate_%']:.1f}%")
print(f"  Opportunity score:     {mkt.iloc[0]['Opportunity_Score']:.4f}")


---
## 11. Conclusions

### Summary of Key Findings

| Finding | Detail |
|---------|--------|
| **Overall Adoption Rate** | 43.3% — over half of diagnosed patients never start treatment |
| **Biggest Drop-off** | Between Diagnosis and Doctor Consultation (46.3% drop) |
| **Strongest Adoption Driver** | Physician recommendation (63.96% vs 33.9% without) |
| **Insurance Impact** | Insured: 48.5% adoption vs Uninsured: 29.3% |
| **Severity Impact** | Severe: 51.4% adoption vs Mild: 38.6% |
| **Top Opportunity Region** | Southeast (3,118 untreated patients) |
| **Model ROC-AUC** | 0.6857 — meaningful predictive signal confirmed |

---

### Key Takeaways for the Business

1. **Physician Recommendation** is the most powerful lever — invest in HCP education and engagement.
2. **Financial Barriers** are real — insurance coverage and affordability gaps must be addressed.
3. **The Southeast Region** presents the highest market opportunity.
4. **Side Effect Concern** is a barrier — patient communication should proactively address this.
5. **High-Severity patients** have higher adoption but are still far from 100% — access barriers persist.

---

### Important Limitations

- This analysis uses **synthetic (fictional) data** — findings are illustrative, not real-world.
- Associations identified are **not causal** — controlled studies are needed to establish causation.
- Simplified assumptions were used in data generation and modelling.
- A real-world analysis would require actual patient claims, prescription, and physician data.

---

> *This project uses synthetic data created for analytical and educational purposes only.
> It does not represent real patients, clinical outcomes, medical advice, or actual pharmaceutical data.*
